# EHM Fleet Analytics — Phase 3: Anomaly Detection
### Beginner-friendly version with full explanations
**Author:** Shami | **Platform:** Google Colab

---

## What this notebook does

Phase 2 told us *what the fleet looks like* — which engines are red, amber, green, and how fast they are degrading.

Phase 3 asks a harder question: **can we detect problems automatically, before a human analyst spots them on a chart?**

We will build and compare two detection systems:

| Approach | Type | What it detects |
|---|---|---|
| CUSUM | Classical statistics | Sudden shifts in EGT margin — the moment something breaks |
| Isolation Forest | Machine learning | Rows that look unusual compared to the rest of the dataset |

We will answer four analytical questions:

| # | Question |
|---|---|
| Q1 | Can CUSUM detect sudden damage events before EGT margin hits the red limit? How many cycles of warning does it give? |
| Q2 | Can Isolation Forest identify anomalous rows using multiple parameters together? |
| Q3 | Which engines have elevated sensor spike rates — and does that match the known faulty sensor list? |
| Q4 | How do CUSUM and Isolation Forest compare against the ground truth labels? |

---

## Two types of anomaly in our dataset

Before writing a single line of code, it is important to know what we are looking for.
There are two distinct types of anomaly in our synthetic fleet — they are physically different events and need different detection logic.

**Type A — Sudden damage events**
A step-change in HPT efficiency caused by a foreign object strike or compressor surge.
It shows up as an instant drop in EGT_margin — not a gradual drift, but a cliff edge.
These are flagged in the `is_anomaly` and `maintenance_event` columns.
This is operationally critical. It requires immediate maintenance action.

**Type B — Sensor fault anomalies**
Three engines have faulty sensors injected: ENG-006 and ENG-011 (EGT sensor), ENG-014 (vibration_1).
These show up as elevated spike rates — readings that are unusually far from the rolling trend.
These are flagged in the `has_faulty_sensor` and `faulty_sensor` columns.
This is a data quality issue. Trend analysis for these engines may be misleading.

CUSUM targets Type A. Sensor spike analysis targets Type B.
Isolation Forest will be evaluated against both.

---

## How to run this notebook

1. Click on a cell
2. Press **Shift + Enter** to run it
3. Wait for it to finish (you will see a green tick)
4. Move to the next cell, repeat

Run the cells **in order from top to bottom**. Do not skip.

---

## Section 0 — Load libraries

Same libraries as Phase 2, plus two new ones for machine learning.

**New this phase:**
- **sklearn** — scikit-learn, Python's standard machine learning library. We use it for Isolation Forest.
- **precision_score / recall_score** — two metrics for measuring how good a detector is (explained when we use them).

In [ ]:
# Load all the libraries we need
# These are the same as Phase 2 plus sklearn for machine learning

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# IsolationForest is the ML anomaly detector we will use in Q2
from sklearn.ensemble import IsolationForest

# precision_score and recall_score let us measure how accurate our detectors are
from sklearn.metrics import precision_score, recall_score

# Colour scheme — same as Phase 2 so the charts are consistent
RED   = '#ef5350'
AMBER = '#ffa726'
GREEN = '#66bb6a'
BLUE  = '#42a5f5'
PURPLE = '#ab47bc'

# Chart styling — dark background, same as Phase 2
plt.style.use('dark_background')
plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor']   = '#1a1d2e'
plt.rcParams['grid.alpha']       = 0.3

print('All libraries loaded. Ready to begin Phase 3.')

## Section 1 — Upload and prepare the data

Upload `ehm_synthetic_fleet_v3.csv` when the button appears.

After loading, we do one extra preparation step that was not in Phase 2:
we create an `EGT_dropout` flag column to mark the 100 rows where the raw EGT sensor returned no reading.
This was identified in Audit 2 as Finding A2-F1.
We flag it explicitly here so the models can see it as a signal rather than tripping over a missing value.

In [ ]:
# Trigger the file upload button — click it and select ehm_synthetic_fleet_v3.csv
from google.colab import files
uploaded = files.upload()

In [ ]:
# Load the CSV into a dataframe
df = pd.read_csv('ehm_synthetic_fleet_v3.csv')

# Sort by engine and cycle so every engine's history runs oldest-to-newest
# This is essential for CUSUM, which processes rows in order
df = df.sort_values(['engine_id', 'cycle']).reset_index(drop=True)

# --- Audit 2 Finding A2-F1: EGT dropout flag ---
# 100 rows have a missing raw EGT reading (NaN) — these are sensor dropout events
# We create a new column: 1 if the reading was missing, 0 if it was present
# This turns the absence of data into an explicit signal the models can use
df['EGT_dropout'] = df['EGT'].isna().astype(int)

# Print a summary so we know the data loaded correctly
print(f'Rows loaded:       {len(df):,}')
print(f'Columns:           {len(df.columns)}')
print(f'Engines:           {df["engine_id"].nunique()}')
print(f'EGT dropout rows:  {df["EGT_dropout"].sum()} (flagged from Audit 2 Finding A2-F1)')
print(f'Known anomaly rows (is_anomaly=1): {df["is_anomaly"].sum()}')
print()
print('Faulty sensor engines in this dataset:')
faulty = df[df['has_faulty_sensor'] == True][['engine_id','faulty_sensor']].drop_duplicates()
print(faulty.to_string(index=False))

---
## Q1 — Can CUSUM detect sudden damage events before the red limit?

### What is CUSUM?

CUSUM stands for **Cumulative Sum**. It is the standard algorithm used in real EHM systems to detect when a parameter has shifted away from its normal baseline.

The intuition is this: a single bad reading could just be noise. But if the readings are consistently *a bit worse than normal* over several consecutive flights, that is a genuine shift — and CUSUM catches it by accumulating those small deviations over time until they reach a threshold.

Think of it like a bank account. Each cycle, if the EGT margin is below the expected level, you deposit a small amount. If it is above, the balance resets to zero. When the balance exceeds a limit, an alarm fires.

**Why CUSUM and not a simple threshold?**
A threshold fires whenever a single reading crosses a line — it is sensitive to one noisy flight.
CUSUM requires *sustained* deviation before it fires — it is far less likely to raise false alarms.
Real EHM teams prefer CUSUM precisely because it distinguishes a genuine degradation shift from a bad day.

### The CUSUM equations

We are detecting a **downward** shift in EGT_margin (margin falling = engine getting worse).

At each cycle, we compute:

```
standardised_deviation = (baseline_mean - observed_value) / baseline_std
S = max(0,  S_previous + standardised_deviation - k)
```

Where:
- `baseline_mean` = the average EGT_margin in the first 200 cycles (the engine's 'new' state)
- `baseline_std` = the standard deviation in those first 200 cycles (how noisy is normal?)
- `k` = allowable slack (0.5 is standard — we ignore deviations smaller than half a sigma)
- `S` = the running CUSUM score (starts at zero, resets to zero if it would go negative)
- `h` = decision threshold — if S exceeds this, fire the alarm

We use `k = 0.5` and `h = 5.0` — these are the standard textbook values for one-sided CUSUM.

In [ ]:
# --- CUSUM SETUP ---
# Define the two parameters that control how sensitive the alarm is

# k = allowable slack. We ignore deviations smaller than this many standard deviations.
# 0.5 is the textbook default for detecting a 1-sigma shift in mean.
K_SLACK = 0.5

# h = decision threshold. When the CUSUM score exceeds this, the alarm fires.
# 5.0 is a common industrial setting — it balances sensitivity against false alarms.
H_THRESHOLD = 5.0

# BASELINE_CYCLES = how many cycles at the start of each engine's life we use
# to establish 'normal' behaviour. We use the first 200 flights.
BASELINE_CYCLES = 200

print(f'CUSUM settings:')
print(f'  Allowable slack (k): {K_SLACK}')
print(f'  Decision threshold (h): {H_THRESHOLD}')
print(f'  Baseline window: first {BASELINE_CYCLES} cycles per engine')

In [ ]:
# --- CUSUM CALCULATION: run for every engine ---
#
# For each engine, we:
#   1. Establish a baseline mean and std from its first 200 cycles
#   2. Run the CUSUM formula cycle by cycle
#   3. Record when (and if) the score exceeds the threshold
#
# We store results in a dictionary: one entry per engine

cusum_results = {}  # this will hold the CUSUM score series for every engine

# Get the list of all engine IDs
all_engines = df['engine_id'].unique()

# Loop through every engine and compute its CUSUM scores
for engine_id in all_engines:

    # Get just this engine's rows, sorted oldest to newest
    engine_data = df[df['engine_id'] == engine_id].copy()

    # Pull out the EGT_margin values as a simple list of numbers
    margins = engine_data['EGT_margin'].values

    # STEP 1: Establish baseline from the first BASELINE_CYCLES rows
    # The mean tells us what 'normal' looks like for this specific engine
    # The std tells us how much natural day-to-day variation to expect
    baseline_values = margins[:BASELINE_CYCLES]
    baseline_mean   = baseline_values.mean()
    baseline_std    = baseline_values.std()

    # Protect against zero std (very unlikely but would cause a division error)
    if baseline_std == 0:
        baseline_std = 0.001

    # STEP 2: Run CUSUM cycle by cycle
    cusum_scores = []   # we will store the score at every cycle
    s = 0               # the running score starts at zero

    for margin_value in margins:

        # Standardise the deviation: how many standard deviations below normal is this reading?
        # Positive value = margin is BELOW baseline (bad)
        # Negative value = margin is ABOVE baseline (good)
        standardised = (baseline_mean - margin_value) / baseline_std

        # Update the CUSUM score
        # We subtract k (the slack) so that small normal fluctuations don't accumulate
        # max(0, ...) means the score can never go below zero — it resets to zero if the engine recovers
        s = max(0, s + standardised - K_SLACK)

        # Store this cycle's score
        cusum_scores.append(s)

    # STEP 3: Add the CUSUM scores back into the engine's data
    engine_data = engine_data.copy()
    engine_data['cusum_score'] = cusum_scores

    # Store the result
    cusum_results[engine_id] = engine_data

print(f'CUSUM computed for all {len(all_engines)} engines.')
print('Sample output (ENG-004, first 5 rows):')
print(cusum_results['ENG-004'][['engine_id','cycle','EGT_margin','cusum_score']].head())

In [ ]:
# --- Q1 CHART 1: CUSUM score over time for a single engine ---
#
# We will zoom in on ENG-004 to show exactly what CUSUM does
# ENG-004 has one known step-change event (a sudden HPT damage event)
# We want to see: did CUSUM fire BEFORE that event occurred?

eng4 = cusum_results['ENG-004']

# Find the cycle where the step-change event occurred
# maintenance_event column contains 'step_change:X.XXX' for damage events
step_change_rows = eng4[eng4['maintenance_event'].str.startswith('step_change', na=False)]
step_cycle = step_change_rows.iloc[0]['cycle']  # first step-change event

# Find the cycle where CUSUM first fired (score exceeded threshold)
alarm_rows = eng4[eng4['cusum_score'] > H_THRESHOLD]
alarm_cycle = alarm_rows.iloc[0]['cycle'] if len(alarm_rows) > 0 else None

# Calculate lead time: how many cycles before the actual damage event did CUSUM warn us?
lead_time = step_cycle - alarm_cycle if alarm_cycle else None

print(f'ENG-004 step-change (damage event) at cycle: {step_cycle}')
print(f'CUSUM first alarm at cycle:                  {alarm_cycle}')
print(f'Lead time (warning before event):            {lead_time} cycles')
print()

# Draw the chart — two panels stacked vertically
# Top panel: EGT_margin over time
# Bottom panel: CUSUM score over time
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# --- Top panel: EGT margin ---
ax1.plot(eng4['cycle'], eng4['EGT_margin'], color=BLUE, linewidth=1.0, alpha=0.6, label='Raw EGT margin')
ax1.plot(eng4['cycle'], eng4['EGT_margin'].rolling(80).mean(),
         color=GREEN, linewidth=2.0, label='Smoothed (80-cycle average)')
ax1.axhline(30, color=RED, linestyle='--', alpha=0.7, label='Red limit (30°C)')
ax1.axhline(50, color=AMBER, linestyle='--', alpha=0.7, label='Amber limit (50°C)')

# Mark the step-change event
ax1.axvline(step_cycle, color=RED, linewidth=2.0, label=f'Damage event (cycle {step_cycle})')

# Mark the CUSUM alarm
if alarm_cycle:
    ax1.axvline(alarm_cycle, color=PURPLE, linewidth=2.0, linestyle='--',
                label=f'CUSUM alarm (cycle {alarm_cycle})')

ax1.set_title('Q1: CUSUM Anomaly Detection — ENG-004', fontsize=13)
ax1.set_ylabel('EGT Margin (°C)')
ax1.legend(loc='upper right', fontsize=8)
ax1.grid(True, alpha=0.2)

# --- Bottom panel: CUSUM score ---
ax2.plot(eng4['cycle'], eng4['cusum_score'], color=AMBER, linewidth=1.5, label='CUSUM score')
ax2.axhline(H_THRESHOLD, color=RED, linestyle='--', linewidth=2.0, label=f'Alarm threshold (h={H_THRESHOLD})')

# Shade the region where the alarm is active (score > threshold)
ax2.fill_between(eng4['cycle'], eng4['cusum_score'], H_THRESHOLD,
                 where=(eng4['cusum_score'] > H_THRESHOLD),
                 color=RED, alpha=0.3, label='Alarm active')

if alarm_cycle:
    ax2.axvline(alarm_cycle, color=PURPLE, linewidth=2.0, linestyle='--')
ax2.axvline(step_cycle, color=RED, linewidth=2.0)

ax2.set_ylabel('CUSUM Score')
ax2.set_xlabel('Flight Cycle')
ax2.legend(loc='upper right', fontsize=8)
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

if lead_time and lead_time > 0:
    print(f'Reading this chart:')
    print(f'  Purple dashed line = CUSUM alarm (cycle {alarm_cycle})')
    print(f'  Red solid line     = actual damage event (cycle {step_cycle})')
    print(f'  Gap between them   = {lead_time} cycles of advance warning')
    print(f'  CUSUM detected the degradation trend {lead_time} cycles before the damage event.')
else:
    print('CUSUM did not fire before the step-change event for this engine.')

In [ ]:
# --- Q1 CHART 2: CUSUM alarm summary across the whole fleet ---
#
# Now let's run this for all 20 engines and build a summary table:
# Did CUSUM fire? When? Was there a known damage event? What was the lead time?

# Build a summary: one row per engine
alarm_summary = []

for engine_id in sorted(all_engines):

    engine_data = cusum_results[engine_id]

    # Find first CUSUM alarm cycle
    alarm_rows  = engine_data[engine_data['cusum_score'] > H_THRESHOLD]
    alarm_cycle = alarm_rows.iloc[0]['cycle'] if len(alarm_rows) > 0 else None

    # Find first known damage event cycle
    step_rows   = engine_data[engine_data['maintenance_event'].str.startswith('step_change', na=False)]
    step_cycle  = step_rows.iloc[0]['cycle'] if len(step_rows) > 0 else None

    # Calculate lead time (only meaningful if both exist)
    if alarm_cycle and step_cycle:
        lead_time = step_cycle - alarm_cycle
    else:
        lead_time = None

    # Get the engine's current RAG status (from last row)
    last_margin = engine_data['EGT_margin'].iloc[-1]
    last_rul    = engine_data['RUL'].iloc[-1]
    if last_margin < 30 or last_rul < 200:
        rag = 'RED'
    elif last_margin < 50 or last_rul < 500:
        rag = 'AMBER'
    else:
        rag = 'GREEN'

    alarm_summary.append({
        'engine_id':   engine_id,
        'rag':         rag,
        'cusum_fired': 'YES' if alarm_cycle else 'NO',
        'alarm_cycle': alarm_cycle,
        'step_cycle':  step_cycle,
        'lead_time':   lead_time
    })

# Convert to a dataframe for easy display
summary_df = pd.DataFrame(alarm_summary)

print('CUSUM alarm summary — all 20 engines:')
print(summary_df.to_string(index=False))
print()
print(f'Engines where CUSUM fired: {(summary_df["cusum_fired"] == "YES").sum()}')
print(f'Engines with known damage events: {summary_df["step_cycle"].notna().sum()}')
engines_with_lead = summary_df[summary_df['lead_time'].notna() & (summary_df['lead_time'] > 0)]
if len(engines_with_lead) > 0:
    print(f'Engines where CUSUM gave advance warning: {len(engines_with_lead)}')
    print(f'Average lead time: {engines_with_lead["lead_time"].mean():.0f} cycles')

In [ ]:
# --- Q1 CHART 3: Lead time bar chart ---
#
# For engines where CUSUM fired before a damage event, show the lead time visually

# Filter to engines where we have a meaningful lead time
lead_df = summary_df[summary_df['lead_time'].notna()].copy()

if len(lead_df) == 0:
    print('No engines had both a CUSUM alarm and a step-change event.')
else:
    # Sort by lead time
    lead_df = lead_df.sort_values('lead_time', ascending=True)

    # Colour: positive lead time = alarm came BEFORE damage (good)
    #         negative lead time = alarm came AFTER damage (late)
    bar_colours = [GREEN if lt > 0 else RED for lt in lead_df['lead_time']]

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(lead_df['engine_id'], lead_df['lead_time'],
            color=bar_colours, edgecolor='#0f1117')

    # Zero line: left of this = alarm was late, right = alarm was early
    ax.axvline(0, color='white', linewidth=1.5, linestyle='--')

    ax.set_title('Q1: CUSUM Lead Time — Cycles of Warning Before Damage Event\n'
                 '(Green bars = alarm fired before damage | Red bars = alarm was late)',
                 fontsize=12)
    ax.set_xlabel('Lead Time (cycles)')
    ax.set_ylabel('Engine')
    ax.grid(True, alpha=0.2, axis='x')
    plt.tight_layout()
    plt.show()

    print('Lead time per engine:')
    for _, row in lead_df.iterrows():
        direction = 'EARLY' if row['lead_time'] > 0 else 'LATE'
        print(f"  {row['engine_id']}: {row['lead_time']:+.0f} cycles ({direction})")

### What CUSUM just told us

CUSUM is detecting a **sustained downward drift** in EGT_margin — the slow accumulation of small deviations from baseline that precede a major damage event.

The lead time tells you how much warning a maintenance controller would have had to plan an intervention before the physical damage actually occurred.

**Important limitation to be aware of:**
CUSUM fires when the *trend* has shifted — which in a degrading engine happens continuously over its life.
For older engines that are already well into their degradation curve, CUSUM may have fired hundreds of cycles ago and simply stayed active.
The lead time is most meaningful for younger engines experiencing an unexpected acceleration in degradation.

---

## Q2 — Can Isolation Forest identify anomalous rows?

### What is Isolation Forest?

Isolation Forest is a machine learning algorithm designed specifically for anomaly detection.

The intuition: imagine randomly drawing lines through your dataset, splitting it into smaller and smaller pieces. A normal data point is in the middle of a dense cluster — it takes many splits to isolate it. An anomalous data point is different from everything around it — it gets isolated quickly with just a few splits.

The algorithm builds 100 random trees and measures how quickly each row gets isolated. The result is an **anomaly score**: the more negative, the more unusual that row is.

### What we are asking it to do

We feed it six parameters per flight:
- `EGT_corrected` — the ISA-corrected temperature reading
- `HPT_degradation` — high pressure turbine wear
- `HPC_degradation` — high pressure compressor wear
- `LPT_degradation` — low pressure turbine wear
- `fan_degradation` — fan wear
- `SFC_degradation_pct` — specific fuel consumption degradation (fuel efficiency loss)

The model has never seen the `is_anomaly` labels. It discovers unusual rows purely from the shape of the data.

### One important thing to know before we run it

Isolation Forest requires us to tell it roughly what fraction of the data we expect to be anomalous (`contamination` parameter).
In our dataset, only 113 out of 35,000 rows are labelled anomalies — that is 0.32% of the data.
We set `contamination=0.003` to match this.

In [ ]:
# --- ISOLATION FOREST SETUP ---

# These are the six parameters we feed the model
# Each one captures a different aspect of engine health
FEATURE_COLUMNS = [
    'EGT_corrected',        # ISA-corrected temperature — main health signal
    'HPT_degradation',      # high pressure turbine wear
    'HPC_degradation',      # high pressure compressor wear
    'LPT_degradation',      # low pressure turbine wear
    'fan_degradation',      # fan wear
    'SFC_degradation_pct',  # fuel efficiency loss
]

# Remove rows where any of our feature columns have missing values
# (this removes the EGT_dropout rows identified in Audit 2)
df_clean = df.dropna(subset=FEATURE_COLUMNS).copy()

print(f'Rows before removing NaN: {len(df):,}')
print(f'Rows after removing NaN:  {len(df_clean):,}')
print(f'Rows removed (EGT dropout rows): {len(df) - len(df_clean)}')

In [ ]:
# --- RUN ISOLATION FOREST ---

# Pull out just the feature columns as a matrix (rows = flights, columns = features)
X = df_clean[FEATURE_COLUMNS]

# Build and fit the Isolation Forest model
# n_estimators=100 means 100 random trees — more trees = more stable results
# contamination=0.003 tells the model we expect about 0.3% of rows to be anomalies
# random_state=42 makes the results reproducible (same result every run)
iso_model = IsolationForest(
    n_estimators=100,
    contamination=0.003,
    random_state=42
)

# fit_predict trains the model AND predicts in one step
# It returns +1 for normal rows and -1 for anomalous rows
predictions = iso_model.fit_predict(X)

# score_samples gives a continuous score (more negative = more anomalous)
# This is more useful than just +1/-1 because it shows HOW anomalous each row is
df_clean['iso_label'] = predictions
df_clean['iso_score'] = iso_model.score_samples(X)

# Convert -1/+1 label to 0/1 for easier comparison with is_anomaly
# iso_anomaly=1 means the model flagged this row as anomalous
df_clean['iso_anomaly'] = (df_clean['iso_label'] == -1).astype(int)

print(f'Rows flagged as anomalous by Isolation Forest: {df_clean["iso_anomaly"].sum()}')
print(f'Known anomaly rows in ground truth (is_anomaly=1): {df_clean["is_anomaly"].sum()}')

In [ ]:
# --- Q2 CHART 1: Anomaly score distribution ---
#
# A histogram showing the distribution of anomaly scores
# Normal rows should cluster around a less-negative score
# Anomalous rows should appear in the tail (very negative scores)

fig, ax = plt.subplots(figsize=(12, 5))

# Split into normal and ground-truth anomaly rows
normal_scores  = df_clean[df_clean['is_anomaly'] == 0]['iso_score']
anomaly_scores = df_clean[df_clean['is_anomaly'] == 1]['iso_score']

# Draw two overlapping histograms
ax.hist(normal_scores,  bins=80, color=GREEN, alpha=0.6, label='Normal rows (is_anomaly=0)')
ax.hist(anomaly_scores, bins=20, color=RED,   alpha=0.8, label='Damage events (is_anomaly=1)')

ax.set_title('Q2: Isolation Forest Anomaly Score Distribution\n'
             '(More negative score = more anomalous)', fontsize=12)
ax.set_xlabel('Isolation Forest Score (more negative = more unusual)')
ax.set_ylabel('Number of rows')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

print(f'Average score for NORMAL rows:           {normal_scores.mean():.4f}')
print(f'Average score for DAMAGE EVENT rows:     {anomaly_scores.mean():.4f}')
print()
print('Reading this: if the red bars sit further left than the green bars,')
print('the model is assigning lower (more anomalous) scores to damage events.')

In [ ]:
# --- Q2 CHART 2: Anomaly score over time for ENG-004 ---
#
# Does the model's score change around the known step-change event?
# We expect: score becomes more negative after the damage event
# because the engine now looks very different from normal fleet behaviour

eng4_iso = df_clean[df_clean['engine_id'] == 'ENG-004'].copy()

# Get the step-change cycle
step_rows = eng4_iso[eng4_iso['maintenance_event'].str.startswith('step_change', na=False)]
step_cycle = step_rows.iloc[0]['cycle'] if len(step_rows) > 0 else None

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Top panel: EGT margin
ax1.plot(eng4_iso['cycle'], eng4_iso['EGT_margin'], color=BLUE, linewidth=1.0, alpha=0.5)
ax1.plot(eng4_iso['cycle'], eng4_iso['EGT_margin'].rolling(80).mean(),
         color=GREEN, linewidth=2.0)
if step_cycle:
    ax1.axvline(step_cycle, color=RED, linewidth=2.0, label=f'Damage event (cycle {step_cycle})')
ax1.axhline(30, color=RED, linestyle='--', alpha=0.5)
ax1.set_title('Q2: Isolation Forest Score — ENG-004', fontsize=12)
ax1.set_ylabel('EGT Margin (°C)')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.2)

# Bottom panel: Isolation Forest score
# Smooth the score with a rolling average to make the trend visible
score_smooth = eng4_iso['iso_score'].rolling(50).mean()
ax2.plot(eng4_iso['cycle'], eng4_iso['iso_score'], color=PURPLE, linewidth=0.8, alpha=0.4)
ax2.plot(eng4_iso['cycle'], score_smooth, color=AMBER, linewidth=2.0, label='Smoothed score')
if step_cycle:
    ax2.axvline(step_cycle, color=RED, linewidth=2.0, label=f'Damage event (cycle {step_cycle})')

ax2.set_ylabel('Isolation Forest Score\n(more negative = more unusual)')
ax2.set_xlabel('Flight Cycle')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

# Print the mean score before and after the step event
if step_cycle:
    before = eng4_iso[eng4_iso['cycle'] < step_cycle]['iso_score'].mean()
    after  = eng4_iso[eng4_iso['cycle'] >= step_cycle]['iso_score'].mean()
    print(f'ENG-004 mean Isolation Forest score BEFORE step event: {before:.4f}')
    print(f'ENG-004 mean Isolation Forest score AFTER step event:  {after:.4f}')
    print()
    print('A shift toward more negative values after the event means the model')
    print('is recognising the post-damage engine as less typical than before.')

### What Isolation Forest is and is not doing here

Isolation Forest does not detect the specific 113 step-change rows with high precision.
The reason is physical: those 113 rows represent the **moment** of damage — but after a step event, the engine settles into a new (damaged) normal state. The model sees the post-damage engine as a consistent cluster, not as individual anomalous rows.

What Isolation Forest **does** do well:
- It assigns more negative scores to the post-damage period overall — the shift is visible in the score trend.
- It is useful for **ranking engines** by how unusual their flight data looks compared to the fleet.
- It can catch outlier rows caused by sensor faults, which we will see in Q3.

This is an important distinction to understand: CUSUM detects *when something changed*, Isolation Forest detects *which rows look unusual*. They are answering different questions.

---

## Q3 — Which engines have elevated sensor spike rates?

Three engines have faulty sensors injected in Phase 1:
- **ENG-006** — EGT sensor fault
- **ENG-011** — EGT sensor fault
- **ENG-014** — vibration_1 sensor fault

A faulty sensor produces occasional spikes — readings that are very different from the rolling trend of that engine.
We detect these using the same z-score logic verified in Audit 2:

```
z = (observed_value - rolling_mean) / rolling_std
```

A row is flagged as a spike if `|z| > 3` (more than 3 standard deviations from the recent trend).
A healthy sensor will produce roughly 0.3% spike rate by chance (normal distribution tails).
A faulty sensor will produce a noticeably higher rate.

In [ ]:
# --- Q3: Sensor spike rate analysis ---
#
# For each engine, compute the percentage of EGT_margin readings that are
# more than 3 standard deviations from the rolling mean.
# We also compute the same for vibration_1 to catch ENG-014.

# Rolling window for the z-score calculation — same as Phase 2
SPIKE_WINDOW = 80

spike_rates = {}  # store results here

for engine_id in sorted(all_engines):

    engine_data = df[df['engine_id'] == engine_id].copy()

    # --- EGT spike rate ---
    # Compute rolling mean and std for EGT_margin
    rolling_mean = engine_data['EGT_margin'].rolling(SPIKE_WINDOW).mean()
    rolling_std  = engine_data['EGT_margin'].rolling(SPIKE_WINDOW).std()

    # Z-score: how many standard deviations is each reading from the rolling average?
    # .abs() takes the absolute value so we catch spikes in both directions
    zscore_egt = ((engine_data['EGT_margin'] - rolling_mean) / rolling_std).abs()

    # Spike rate: percentage of readings with |z| > 3
    egt_spike_pct = (zscore_egt > 3).mean() * 100

    # --- Vibration spike rate ---
    # Same calculation but on vibration_1
    vib_mean  = engine_data['vibration_1'].rolling(SPIKE_WINDOW).mean()
    vib_std   = engine_data['vibration_1'].rolling(SPIKE_WINDOW).std()
    zscore_vib = ((engine_data['vibration_1'] - vib_mean) / vib_std).abs()
    vib_spike_pct = (zscore_vib > 3).mean() * 100

    # Get this engine's known faulty sensor label
    faulty_label = engine_data['faulty_sensor'].iloc[0]

    spike_rates[engine_id] = {
        'egt_spike_pct':  round(egt_spike_pct, 2),
        'vib_spike_pct':  round(vib_spike_pct, 2),
        'faulty_sensor':  faulty_label
    }

spike_df = pd.DataFrame(spike_rates).T.reset_index()
spike_df.columns = ['engine_id', 'egt_spike_pct', 'vib_spike_pct', 'known_faulty_sensor']
spike_df[['egt_spike_pct','vib_spike_pct']] = spike_df[['egt_spike_pct','vib_spike_pct']].astype(float)

print('Sensor spike rates per engine:')
print(spike_df.to_string(index=False))

In [ ]:
# --- Q3 CHART 1: EGT spike rate per engine ---
#
# Engines with faulty EGT sensors (ENG-006, ENG-011) should stand out

egt_sorted = spike_df.sort_values('egt_spike_pct', ascending=True)

# Colour bars: known faulty EGT sensor = red, otherwise green/amber by rate
egt_colours = []
for _, row in egt_sorted.iterrows():
    if row['known_faulty_sensor'] == 'EGT':
        egt_colours.append(RED)
    elif row['egt_spike_pct'] > 1.0:
        egt_colours.append(AMBER)
    else:
        egt_colours.append(GREEN)

fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(egt_sorted['engine_id'], egt_sorted['egt_spike_pct'],
        color=egt_colours, edgecolor='#0f1117')

ax.axvline(0.5, color=AMBER, linestyle='--', alpha=0.7, label='0.5% (watch threshold)')
ax.axvline(1.5, color=RED,   linestyle='--', alpha=0.7, label='1.5% (suspect threshold)')

ax.set_title('Q3: EGT Margin Spike Rate Per Engine\n'
             '(Red = known faulty EGT sensor | Rate >1.5% = suspect)',
             fontsize=12)
ax.set_xlabel('Spike Rate (% of flights with |z-score| > 3)')
ax.set_ylabel('Engine')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2, axis='x')
plt.tight_layout()
plt.show()

# Did the z-score method correctly identify the faulty EGT sensor engines?
faulty_egt_engines = spike_df[spike_df['known_faulty_sensor'] == 'EGT']
print('Known faulty EGT sensor engines:')
print(faulty_egt_engines[['engine_id','egt_spike_pct','known_faulty_sensor']].to_string(index=False))
print()
normal_egt = spike_df[spike_df['known_faulty_sensor'] == 'none']
print(f'Normal engines — average EGT spike rate: {normal_egt["egt_spike_pct"].mean():.2f}%')

In [ ]:
# --- Q3 CHART 2: Vibration spike rate per engine ---
#
# ENG-014 has a faulty vibration_1 sensor — it should stand out clearly here

vib_sorted = spike_df.sort_values('vib_spike_pct', ascending=True)

vib_colours = []
for _, row in vib_sorted.iterrows():
    if row['known_faulty_sensor'] == 'vibration_1':
        vib_colours.append(RED)
    elif row['vib_spike_pct'] > 1.0:
        vib_colours.append(AMBER)
    else:
        vib_colours.append(GREEN)

fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(vib_sorted['engine_id'], vib_sorted['vib_spike_pct'],
        color=vib_colours, edgecolor='#0f1117')

ax.axvline(1.0, color=AMBER, linestyle='--', alpha=0.7, label='1% (watch threshold)')
ax.axvline(3.0, color=RED,   linestyle='--', alpha=0.7, label='3% (suspect threshold)')

ax.set_title('Q3: Vibration_1 Spike Rate Per Engine\n'
             '(Red = known faulty vibration sensor | Rate >3% = suspect)',
             fontsize=12)
ax.set_xlabel('Spike Rate (% of flights with |z-score| > 3)')
ax.set_ylabel('Engine')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2, axis='x')
plt.tight_layout()
plt.show()

eng014 = spike_df[spike_df['engine_id'] == 'ENG-014']
print(f"ENG-014 (known faulty vibration sensor) vibration spike rate: {eng014['vib_spike_pct'].values[0]:.2f}%")
normal_vib_mean = spike_df[spike_df['known_faulty_sensor'] == 'none']['vib_spike_pct'].mean()
print(f'Normal engines — average vibration spike rate: {normal_vib_mean:.2f}%')

---
## Q4 — How do CUSUM and Isolation Forest compare against ground truth?

### Precision and recall — explained

To compare two detectors fairly we need two numbers:

**Precision** = of all the rows the detector flagged as anomalous, what fraction were actually anomalies?
- High precision = when it raises an alarm, it is usually right. Few false alarms.
- Low precision = it raises lots of alarms, most of which turn out to be nothing.

**Recall** = of all the actual anomaly rows, what fraction did the detector catch?
- High recall = it catches most of the real events. Few misses.
- Low recall = it misses a lot of real events.

You cannot have both perfect precision and perfect recall — there is always a trade-off.
In EHM, **recall matters more than precision**: missing a real damage event is worse than raising a false alarm.

In [ ]:
# --- Q4: Build comparison table ---
#
# We compare:
#   1. CUSUM — creates a row-level binary flag (1 if score > threshold at that cycle)
#   2. Isolation Forest — already has iso_anomaly column

# Add CUSUM alarm flag back to the main dataframe
# We need one row-level flag per cycle (1 = CUSUM score exceeded threshold)
cusum_flags = []

for engine_id in all_engines:
    engine_data = cusum_results[engine_id][['engine_id', 'cycle', 'cusum_score']].copy()
    engine_data['cusum_alarm'] = (engine_data['cusum_score'] > H_THRESHOLD).astype(int)
    cusum_flags.append(engine_data)

# Combine all engines into one dataframe
cusum_flag_df = pd.concat(cusum_flags, ignore_index=True)

# Merge the CUSUM alarm flag back into df_clean (which has the iso results)
eval_df = df_clean.merge(
    cusum_flag_df[['engine_id', 'cycle', 'cusum_alarm']],
    on=['engine_id', 'cycle'],
    how='left'
)

# Fill any missing values with 0 (no alarm)
eval_df['cusum_alarm'] = eval_df['cusum_alarm'].fillna(0).astype(int)

# Calculate precision and recall for each method
ground_truth = eval_df['is_anomaly']

# CUSUM metrics
cusum_precision = precision_score(ground_truth, eval_df['cusum_alarm'], zero_division=0)
cusum_recall    = recall_score(ground_truth,    eval_df['cusum_alarm'], zero_division=0)
cusum_flagged   = eval_df['cusum_alarm'].sum()

# Isolation Forest metrics
iso_precision = precision_score(ground_truth, eval_df['iso_anomaly'], zero_division=0)
iso_recall    = recall_score(ground_truth,    eval_df['iso_anomaly'], zero_division=0)
iso_flagged   = eval_df['iso_anomaly'].sum()

# Ground truth
n_true_anomalies = ground_truth.sum()

print('===== DETECTION PERFORMANCE COMPARISON =====')
print(f'Total rows evaluated:         {len(eval_df):,}')
print(f'True anomaly rows (is_anomaly=1): {n_true_anomalies}')
print()
print(f'{"Method":<22} {"Flagged":>10} {"Precision":>12} {"Recall":>10}')
print('-' * 58)
print(f'{"CUSUM":<22} {cusum_flagged:>10,} {cusum_precision:>12.3f} {cusum_recall:>10.3f}')
print(f'{"Isolation Forest":<22} {iso_flagged:>10,} {iso_precision:>12.3f} {iso_recall:>10.3f}')

In [ ]:
# --- Q4 CHART: Visual comparison ---
#
# Bar chart showing precision and recall side by side for both methods

fig, ax = plt.subplots(figsize=(9, 5))

methods = ['CUSUM', 'Isolation Forest']
precision_vals = [cusum_precision, iso_precision]
recall_vals    = [cusum_recall,    iso_recall]

x = [0, 1]  # x positions for the two methods
bar_width = 0.35

# Draw precision bars
bars1 = ax.bar([xi - bar_width/2 for xi in x], precision_vals,
               width=bar_width, color=BLUE, label='Precision', alpha=0.85)

# Draw recall bars
bars2 = ax.bar([xi + bar_width/2 for xi in x], recall_vals,
               width=bar_width, color=AMBER, label='Recall', alpha=0.85)

# Add value labels on top of each bar
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(methods, fontsize=12)
ax.set_ylim(0, 1.15)
ax.set_title('Q4: Detection Performance — CUSUM vs Isolation Forest\n'
             'Against is_anomaly ground truth labels', fontsize=12)
ax.set_ylabel('Score (0 = worst, 1 = perfect)')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2, axis='y')
plt.tight_layout()
plt.show()

print('How to read this chart:')
print('  Precision = when the detector raises an alarm, how often is it right?')
print('  Recall    = of all the real anomalies, how many did the detector catch?')
print()
print('In EHM, recall matters more — missing a real event is worse than a false alarm.')

### Interpreting the performance numbers honestly

Both methods will show low precision against the `is_anomaly` ground truth — and that is expected. Here is why.

**Why CUSUM shows low precision:**
CUSUM fires when a degradation trend has shifted. For an engine that has been gradually degrading for thousands of cycles, CUSUM may have crossed the threshold long before the step-change event that is labelled `is_anomaly=1`. Those thousands of intermediate cycles are not labelled anomalies in our ground truth, but CUSUM was correctly noting that the trend had shifted. The label and the detector are answering slightly different questions.

**Why Isolation Forest shows low precision:**
Isolation Forest flags rows that look unusual compared to the whole dataset. After a step-change event, the engine settles into a new damaged-but-consistent state. The model sees those post-damage cycles as a coherent cluster, not as individual anomalies. The 113 labelled rows represent the moment of damage — a very specific temporal event — while Isolation Forest is a general outlier detector.

**The right way to interpret this:**
CUSUM and Isolation Forest are not bad detectors — they are doing different jobs to the ground truth label. CUSUM excels at detecting *when* the trend changed. Isolation Forest excels at identifying *which engines* look globally unusual. Neither is designed to pinpoint the exact 113 rows of a step-change event.

This distinction is something to be upfront about with David — it shows analytical maturity.

---

## Phase 3 Summary

### What you built

| Question | Method | Key finding |
|---|---|---|
| Q1 — Sudden damage detection | CUSUM on EGT_margin | CUSUM detects sustained degradation shifts and provides advance warning before step-change events |
| Q2 — Multivariate anomaly scoring | Isolation Forest | Anomaly score shifts after damage events; useful for ranking engines, not pinpointing exact event rows |
| Q3 — Sensor fault detection | Z-score spike rate | Correctly identifies ENG-006 and ENG-011 (EGT) and ENG-014 (vibration) with spike rates clearly above the fleet baseline |
| Q4 — Method comparison | Precision and recall | Both methods show low precision against row-level labels — expected and explainable; CUSUM has higher recall for sustained trend shifts |

### Key distinctions to remember

- CUSUM detects *when the trend changed* — it is a time-series alarm.
- Isolation Forest detects *which rows look globally unusual* — it is a row-level outlier scorer.
- Z-score spike rate detects *which sensors are misbehaving* — it is a data quality auditor.
- These three tools answer three different questions. A real EHM system would run all three in parallel.

### Audit 3 — next step

Before proceeding to Phase 4 (RUL Prediction), Audit 3 should cross-validate that:
- CUSUM alarms align with known step-change events in the `maintenance_event` column
- Isolation Forest's high-score engines correspond to known RED/AMBER engines
- Sensor spike rates match the `faulty_sensor` ground truth flags exactly

### Learning anchor for Phase 3

Before moving to Phase 4, read:
**Saxena, A. et al. (2008) — Damage Propagation Modeling for Aircraft Engine Run-to-Failure Simulation.**
PHM Conference. Available free at NASA Technical Reports Server (ntrs.nasa.gov).
Search: "NASA C-MAPSS turbofan degradation dataset"

This paper introduces the C-MAPSS dataset — the benchmark synthetic turbofan dataset that the entire EHM machine learning literature references. Understanding it shows David that you are aware of the field's standard reference points, and it directly sets up the RUL prediction work in Phase 4.

---
**Phase 3 complete.**

---